# Combine time-lapse chunks → one OME-TIFF hyperstack

Point this at a folder of time-lapse pieces (`.nd2`, `.btf`, `.tif/.tiff`, `.ome.tif`). It:

1. Finds the image files (auto-picks the most common type, or you choose one)
2. Opens each one **lazily** (nothing is loaded in full)
3. Checks that they can be stacked (same C / Z / Y / X / dtype / pixel size)
4. Concatenates them along **T**, streaming **one timepoint at a time** to disk, so RAM use stays at about one timepoint
5. Writes a **BigTIFF OME-TIFF** (`TCZYX`) with pixel sizes, channel names, and per-plane timestamps (`DeltaT`) in the OME-XML
6. Keeps **all original metadata** from every source file, both embedded in the TIFF (private tag 65000, JSON) and in a `*_metadata.json` sidecar, plus a provenance table (which output frames came from which file)

**Opening in Fiji:** use *File → Import → Bio-Formats* and tick **"Use virtual stack"** so Fiji doesn't load the whole file into RAM. **Don't re-save from Fiji.** Re-saving is the step that drops the metadata. Treat this output as the master file.

**Requirements** (one time, in your conda env):
`conda install -c conda-forge tifffile imagecodecs zarr dask pandas` and `pip install nd2`

## 1 · Settings: edit this cell

In [ ]:
from pathlib import Path

INPUT_DIR   = Path(r"C:\path\to\timelapse_folder")   # folder holding the chunks
FILE_TYPE   = "auto"      # "auto" | ".nd2" | ".btf" | ".tif" | ".tiff" | ".ome.tif"
SORT_BY     = "name"      # "name" = natural sort (t2 before t10) | "acq_time" = acquisition start time from metadata
OUTPUT_PATH = None        # None → <INPUT_DIR>/<folder name>_combined.ome.tif
ND2_POSITION = 0          # if a .nd2 has multiple XY positions (P), which one to export
COMPRESSION  = None       # None (fastest, biggest) | "zlib" (lossless, smaller, slower)
EMBED_VENDOR_METADATA = True   # embed full original metadata JSON inside the TIFF (sidecar JSON is always written)
RECURSIVE   = False       # also search subfolders

## 2 · Imports and helpers

In [ ]:
import json, re, math, datetime as dt, shutil, warnings
from dataclasses import dataclass, field, is_dataclass, asdict
from collections import Counter
import numpy as np, pandas as pd
import tifffile, zarr, dask.array as da

try:
    import nd2
except ImportError:
    nd2 = None
    print("nd2 package not installed; .nd2 files can't be read (pip install nd2)")

SUPPORTED = [".ome.tif", ".ome.tiff", ".nd2", ".btf", ".tif", ".tiff", ".tf8"]

def ext_of(p: Path) -> str:
    n = p.name.lower()
    for e in SUPPORTED:          # longest-first so .ome.tif beats .tif
        if n.endswith(e):
            return e
    return p.suffix.lower()

def natural_key(p: Path):
    return [int(s) if s.isdigit() else s.lower() for s in re.split(r"(\d+)", p.name)]

def jsonable(o, _depth=0):
    """Recursively convert vendor metadata objects to JSON-safe types (keeps everything, loses nothing silently)."""
    if _depth > 50: return str(o)
    if is_dataclass(o) and not isinstance(o, type): o = asdict(o)
    if isinstance(o, dict):  return {str(k): jsonable(v, _depth+1) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)): return [jsonable(v, _depth+1) for v in o]
    if isinstance(o, np.ndarray): return o.tolist() if o.size < 100_000 else f"<ndarray {o.shape} {o.dtype}>"
    if isinstance(o, np.generic): return o.item()
    if isinstance(o, (dt.datetime, dt.date)): return o.isoformat()
    if isinstance(o, bytes): return o.decode("latin-1", "replace") if len(o) < 10_000 else f"<{len(o)} bytes>"
    if isinstance(o, (str, int, float, bool)) or o is None: return o
    if hasattr(o, "_asdict"): return jsonable(o._asdict(), _depth+1)
    return str(o)

def parse_dt(s):
    if not s: return None
    s = str(s).strip()
    for fmt in (None, "%d/%m/%Y  %H:%M:%S", "%m/%d/%Y  %I:%M:%S %p", "%d.%m.%Y  %H:%M:%S", "%Y-%m-%d %H:%M:%S"):
        try:
            return dt.datetime.fromisoformat(s.replace("Z", "+00:00")) if fmt is None else dt.datetime.strptime(s, fmt)
        except Exception:
            pass
    m = re.search(r"\d{1,2}/\d{1,2}/\d{4}\s+\d{1,2}:\d{2}:\d{2}(\s*[AP]M)?", s)   # NIS "Date:" inside free text
    if m:
        return parse_dt(re.sub(r"\s+", "  ", m.group(0), count=1))
    return None

@dataclass
class Source:
    path: Path
    data: "da.Array"                 # lazy, always TCZYX
    pixel_size: tuple                # (x, y, z) in µm, None where unknown
    channel_names: list
    rel_times: list | None           # seconds from this file's first frame, one per T (None if unknown)
    start_time: dt.datetime | None   # absolute acquisition start, if known
    metadata: dict = field(default_factory=dict)   # ALL original metadata, JSON-safe
    _handle: object = None           # keep the file open while the lazy array is used

    @property
    def T(self): return self.data.shape[0]

def to_tczyx(arr, axes: str):
    """Reorder an array with arbitrary axis letters into TCZYX, adding singleton axes where missing."""
    axes = axes.upper()
    alias = {"I": "T", "Q": "T", "S": "C", "P": "P"}     # tifffile generic axes → T, RGB samples → C
    norm = []
    for a in axes:
        b = a if a in "TCZYXP" else alias.get(a, a)
        if b in norm:                                    # e.g. both C and S present: keep extra as-is
            b = a
        norm.append(b)
    extra = [i for i, a in enumerate(norm) if a not in "TCZYX"]
    for i in extra:
        if arr.shape[i] != 1:
            raise ValueError(f"Can't map axis '{axes[i]}' (size {arr.shape[i]}) in '{axes}' to TCZYX")
    keep = [i for i in range(len(norm)) if i not in extra]
    arr = arr[tuple(slice(None) if i in keep else 0 for i in range(len(norm)))]
    norm = [norm[i] for i in keep]
    for a in "TCZYX":
        if a not in norm:
            arr = arr[None]; norm.insert(0, a)
    return arr.transpose([norm.index(a) for a in "TCZYX"])

## 3 · Readers: one per format, each returns a lazy `Source`

In [ ]:
def read_nd2(p: Path) -> Source:
    if nd2 is None:
        raise ImportError("pip install nd2")
    f = nd2.ND2File(p)
    sizes = dict(f.sizes)                                # e.g. {'T':10,'P':2,'Z':5,'C':3,'Y':..,'X':..}
    arr = f.to_dask()
    axes = "".join(sizes.keys())
    if "P" in axes:
        arr = arr[tuple(ND2_POSITION if a == "P" else slice(None) for a in axes)]
        axes = axes.replace("P", "")
    data = to_tczyx(arr, axes)

    vs = f.voxel_size()
    try:
        chans = [c.channel.name for c in f.metadata.channels]
    except Exception:
        chans = [f"C{i}" for i in range(data.shape[1])]

    # per-T relative time from frame metadata (first frame of each timepoint)
    rel = None
    try:
        rel = [None] * data.shape[0]
        for seq, idx in enumerate(f.loop_indices):
            t = idx.get("T", 0)
            if idx.get("P", 0) != ND2_POSITION and "P" in sizes: continue
            if rel[t] is None:
                rel[t] = f.frame_metadata(seq).channels[0].time.relativeTimeMs / 1000.0
        if any(r is None for r in rel): rel = None
        else: rel = [r - rel[0] for r in rel]
    except Exception as e:
        rel = None

    ti = f.text_info or {}
    start = parse_dt(ti.get("date")) or parse_dt(ti.get("description"))

    meta = {"format": "nd2", "sizes": sizes, "position_exported": ND2_POSITION if "P" in sizes else None}
    for name, getter in [("attributes", lambda: f.attributes), ("metadata", lambda: f.metadata),
                         ("experiment", lambda: f.experiment), ("text_info", lambda: f.text_info),
                         ("rois", lambda: f.rois), ("events", lambda: f.events()),
                         ("unstructured_metadata", lambda: f.unstructured_metadata())]:
        try:
            meta[name] = jsonable(getter())
        except Exception as e:
            meta[name] = f"<could not read: {e}>"
    return Source(p, data, (vs.x, vs.y, vs.z), chans, rel, start, meta, f)


def read_tiff(p: Path) -> Source:
    tf = tifffile.TiffFile(p)
    s = tf.series[0]
    try:                                                 # lazy, chunked by page
        z = zarr.open(s.aszarr(), mode="r")
        if isinstance(z, zarr.Group): z = z[0]          # pyramidal → full-res level
        arr = da.from_zarr(z)
    except Exception:
        arr = da.from_array(tifffile.memmap(p), chunks="auto")   # uncompressed fallback
    data = to_tczyx(arr, s.axes)

    meta = {"format": "tiff", "series_axes": s.axes, "series_shape": list(s.shape),
            "is_bigtiff": tf.is_bigtiff, "flags": sorted(f for f in tf.flags)}
    px = [None, None, None]; chans = None; rel = None; start = None
    page_tags = {}
    for tag in tf.pages[0].tags:                         # first-page tags (skip the huge offset tables)
        if tag.name in ("StripOffsets", "StripByteCounts", "TileOffsets", "TileByteCounts"): continue
        page_tags[tag.name] = jsonable(tag.value)
    meta["first_page_tags"] = page_tags

    if tf.ome_metadata:
        meta["ome_xml"] = tf.ome_metadata
        try:
            om = tifffile.xml2dict(tf.ome_metadata)["OME"]
            img = om["Image"][0] if isinstance(om["Image"], list) else om["Image"]
            pix = img["Pixels"]
            px = [pix.get("PhysicalSizeX"), pix.get("PhysicalSizeY"), pix.get("PhysicalSizeZ")]
            ch = pix.get("Channel", [])
            ch = ch if isinstance(ch, list) else [ch]
            chans = [c.get("Name", f"C{i}") for i, c in enumerate(ch)] or None
            start = parse_dt(img.get("AcquisitionDate"))
            planes = pix.get("Plane", [])
            planes = planes if isinstance(planes, list) else [planes]
            tt = {}
            for pl in planes:
                if pl.get("TheC", 0) == 0 and pl.get("TheZ", 0) == 0 and "DeltaT" in pl:
                    tt[pl.get("TheT", 0)] = float(pl["DeltaT"])
            if len(tt) == data.shape[0]:
                rel = [tt[t] - tt[0] for t in sorted(tt)]
        except Exception as e:
            meta["ome_parse_error"] = str(e)
    if tf.imagej_metadata:
        ij = tf.imagej_metadata
        meta["imagej_metadata"] = jsonable(ij)
        if px[0] is None and "XResolution" in tf.pages[0].tags:
            num, den = tf.pages[0].tags["XResolution"].value
            if num: px[0] = px[1] = den / num
        if px[2] is None and "spacing" in ij: px[2] = ij["spacing"]
        if rel is None and ij.get("finterval"):
            rel = [i * float(ij["finterval"]) for i in range(data.shape[0])]
        if chans is None and isinstance(ij.get("Labels"), (list, tuple)) and len(ij["Labels"]) == data.shape[1]:
            chans = list(ij["Labels"])
    for attr in ("micromanager_metadata", "shaped_metadata", "fluoview_metadata", "lsm_metadata",
                 "metaseries_metadata", "nih_metadata", "scanimage_metadata"):
        try:
            v = getattr(tf, attr, None)
            if v: meta[attr] = jsonable(v)
        except Exception:
            pass
    chans = chans or [f"C{i}" for i in range(data.shape[1])]
    return Source(p, data, tuple(px), chans, rel, start, meta, tf)


READERS = {".nd2": read_nd2}
for e in (".ome.tif", ".ome.tiff", ".btf", ".tif", ".tiff", ".tf8"):
    READERS[e] = read_tiff

## 4 · Find and open the files

In [ ]:
files = [p for p in (INPUT_DIR.rglob("*") if RECURSIVE else INPUT_DIR.iterdir())
         if p.is_file() and ext_of(p) in READERS and "_combined" not in p.name]
counts = Counter(ext_of(p) for p in files)
print("Found:", dict(counts) or "nothing")
if not files:
    raise FileNotFoundError(f"No supported image files in {INPUT_DIR}")

chosen = FILE_TYPE.lower() if FILE_TYPE != "auto" else counts.most_common(1)[0][0]
if FILE_TYPE == "auto" and len(counts) > 1:
    print(f"⚠ Multiple types present. Using the most common: {chosen}. Set FILE_TYPE to override.")
files = [p for p in files if ext_of(p) == chosen]

sources = [READERS[chosen](p) for p in files]
if SORT_BY == "acq_time":
    if all(s.start_time for s in sources):
        sources.sort(key=lambda s: s.start_time)
    else:
        print("⚠ Not every file has an acquisition time. Falling back to natural name sort.")
        sources.sort(key=lambda s: natural_key(s.path))
else:
    sources.sort(key=lambda s: natural_key(s.path))

pd.DataFrame([{
    "file": s.path.name, "T": s.data.shape[0], "C": s.data.shape[1], "Z": s.data.shape[2],
    "Y": s.data.shape[3], "X": s.data.shape[4], "dtype": str(s.data.dtype),
    "px_x_um": s.pixel_size[0], "px_z_um": s.pixel_size[2], "channels": ", ".join(map(str, s.channel_names)),
    "start": s.start_time, "has_timestamps": s.rel_times is not None,
} for s in sources])

## 5 · Check that the chunks are compatible
A hard error means the files can't be stacked. Warnings are worth a look but don't block the write.

In [ ]:
ref = sources[0]
problems, warns = [], []
for s in sources[1:]:
    if s.data.shape[1:] != ref.data.shape[1:]:
        problems.append(f"{s.path.name}: CZYX {s.data.shape[1:]} ≠ {ref.data.shape[1:]}")
    if s.data.dtype != ref.data.dtype:
        problems.append(f"{s.path.name}: dtype {s.data.dtype} ≠ {ref.data.dtype}")
    for i, ax in enumerate("XYZ"):
        a, b = s.pixel_size[i], ref.pixel_size[i]
        if a and b and not math.isclose(a, b, rel_tol=1e-3):
            warns.append(f"{s.path.name}: pixel size {ax} {a} ≠ {b}")
    if list(s.channel_names) != list(ref.channel_names):
        warns.append(f"{s.path.name}: channel names {s.channel_names} ≠ {ref.channel_names}")
if problems:
    raise ValueError("Files can't be combined:\n  " + "\n  ".join(problems))
print("✓ shapes and dtypes match" + ("" if warns else "; metadata consistent"))
for w in warns: print("⚠", w)

## 6 · Build the combined timeline and provenance

In [ ]:
# Timestamps for the combined T axis (seconds from the first frame of the first file).
# Priority: absolute start times (exact gaps between chunks) > per-file relative times
# chained with the median frame interval > none.
T_total = sum(s.T for s in sources)
times, time_method = [], None
if all(s.start_time and s.rel_times for s in sources):
    t0 = sources[0].start_time
    for s in sources:
        off = (s.start_time - t0).total_seconds()
        times += [off + r for r in s.rel_times]
    time_method = "absolute start time + per-frame relative time"
elif all(s.rel_times for s in sources):
    diffs = np.concatenate([np.diff(s.rel_times) for s in sources if len(s.rel_times) > 1] or [[0]])
    gap = float(np.median(diffs)) if diffs.size else 0.0
    off = 0.0
    for s in sources:
        times += [off + r for r in s.rel_times]
        off = times[-1] + gap
    time_method = f"chained relative times; gaps between files ASSUMED = median interval ({gap:.3g} s)"
else:
    times = None
    time_method = "no timestamps found in sources"
print("Timestamps:", time_method)

prov, t = [], 0
for s in sources:
    prov.append({"file": s.path.name, "out_T_first": t, "out_T_last": t + s.T - 1, "n_T": s.T,
                 "start_time": s.start_time.isoformat() if s.start_time else None})
    t += s.T
pd.DataFrame(prov)

## 7 · Write the OME-TIFF (streams one timepoint at a time)

In [ ]:
out = Path(OUTPUT_PATH) if OUTPUT_PATH else INPUT_DIR / f"{INPUT_DIR.name}_combined.ome.tif"
_, C, Z, Y, X = ref.data.shape
shape = (T_total, C, Z, Y, X)
nbytes = int(np.prod(shape)) * ref.data.dtype.itemsize
free = shutil.disk_usage(out.parent).free
print(f"Output {shape} {ref.data.dtype} ≈ {nbytes/1e9:.2f} GB uncompressed | free on disk: {free/1e9:.1f} GB")
if COMPRESSION is None and nbytes > free * 0.95:
    raise OSError("Not enough disk space. Set COMPRESSION='zlib' or pick another OUTPUT_PATH.")

# ---- OME metadata (what Fiji/Bio-Formats, napari, QuPath etc. read) ----
ome = {"axes": "TCZYX", "Name": out.name.replace(".ome.tif", ""),
       "Channel": {"Name": [str(c) for c in ref.channel_names]}}
for k, v in zip(("PhysicalSizeX", "PhysicalSizeY", "PhysicalSizeZ"), ref.pixel_size):
    if v: ome[k] = float(v); ome[k + "Unit"] = "µm"
if times:
    if T_total > 1: ome["TimeIncrement"] = float(np.median(np.diff(times))); ome["TimeIncrementUnit"] = "s"
    ome["Plane"] = {"DeltaT": [float(times[t]) for t in range(T_total) for _ in range(C * Z)],
                    "DeltaTUnit": ["s"] * (T_total * C * Z)}

# ---- full original metadata, kept with the file ----
full_meta = {
    "created": dt.datetime.now().isoformat(), "tool": "combine_timelapse notebook",
    "tifffile_version": tifffile.__version__, "output_shape_TCZYX": list(shape),
    "dtype": str(ref.data.dtype), "timestamp_method": time_method,
    "timestamps_s": times, "provenance": prov,
    "sources": [{"file": str(s.path), **s.metadata} for s in sources],
}
sidecar = out.with_name(out.name.replace(".ome.tif", "") + "_metadata.json")
sidecar.write_text(json.dumps(full_meta, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
print("Sidecar:", sidecar)

extratags = []
if EMBED_VENDOR_METADATA:
    blob = json.dumps(full_meta, ensure_ascii=True, default=str)
    extratags = [(65000, "s", 0, blob, True)]     # private tag; read back with tf.pages[0].tags[65000].value

def planes():
    for s in sources:
        for ti in range(s.T):
            block = np.asarray(s.data[ti].compute())   # one timepoint (C,Z,Y,X) in RAM
            for c in range(C):
                for z in range(Z):
                    yield block[c, z]
            print(f"\r  {s.path.name}  T {ti+1}/{s.T}   ", end="")

def write(meta):
    with tifffile.TiffWriter(out, bigtiff=True, ome=True) as tw:
        tw.write(planes(), shape=shape, dtype=ref.data.dtype, metadata=meta,
                 compression=COMPRESSION, photometric="minisblack", extratags=extratags)

try:
    write(ome)
except (ValueError, TypeError) as e:          # very old tifffile without Plane support
    print("\n⚠ Plane timestamps rejected by tifffile:", e, "→ writing without per-plane DeltaT")
    ome.pop("Plane", None); write(ome)
print(f"\n✓ Wrote {out}  ({out.stat().st_size/1e9:.2f} GB)")

## 8 · Verify the output
*Informational check, not a fix.* It re-opens the written file lazily, confirms shape and metadata, and compares random planes pixel by pixel against the source files. Rerun it whenever you want to double-check a file.

In [ ]:
# DEBUG / VERIFY (informational, one-off per output file): confirms the combined file
# matches the sources (shape, pixel data at random planes, OME metadata, embedded JSON). Fixes nothing.
rng = np.random.default_rng(0)
with tifffile.TiffFile(out) as tf:
    s_out = tf.series[0]
    print("axes:", s_out.axes, "shape:", s_out.shape, "dtype:", s_out.dtype, "| OME:", tf.is_ome)
    zout = to_tczyx(da.from_zarr(zarr.open(s_out.aszarr(), mode="r")), s_out.axes)   # tifffile hides size-1 axes
    assert tuple(zout.shape) == shape, f"shape mismatch {zout.shape} vs {shape}"
    for _ in range(min(8, T_total * C * Z)):
        T_ = int(rng.integers(T_total)); c = int(rng.integers(C)); z = int(rng.integers(Z))
        row = next(p for p in prov if p["out_T_first"] <= T_ <= p["out_T_last"])
        src = next(s for s in sources if s.path.name == row["file"])
        a = np.asarray(zout[T_, c, z].compute()); b = np.asarray(src.data[T_ - row["out_T_first"], c, z].compute())
        assert np.array_equal(a, b), f"pixel mismatch at T={T_} C={c} Z={z}"
    print("✓ random planes identical to source")
    om = tifffile.xml2dict(tf.ome_metadata)["OME"]["Image"]
    pix = (om[0] if isinstance(om, list) else om)["Pixels"]
    print("PhysicalSize X/Y/Z:", pix.get("PhysicalSizeX"), pix.get("PhysicalSizeY"), pix.get("PhysicalSizeZ"),
          "| TimeIncrement:", pix.get("TimeIncrement"))
    if EMBED_VENDOR_METADATA:
        emb = json.loads(tf.pages[0].tags[65000].value)
        print("✓ embedded metadata: sources =", [Path(x["file"]).name for x in emb["sources"]])

## Reading the preserved metadata later
```python
import json, tifffile
with tifffile.TiffFile("..._combined.ome.tif") as tf:
    meta = json.loads(tf.pages[0].tags[65000].value)   # everything: per-source vendor metadata, provenance, timestamps
meta["provenance"]                       # which output T came from which file
meta["sources"][0]["text_info"]          # e.g. NIS-Elements text info from the first nd2
```